# Diffusion Posterior Sampling for Astronomical Inverse Problems

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GabrielMissael/diffusion_posterior_tutorial/blob/master/notebooks/diffusion_posterior_sampling.ipynb)
[![View on GitHub](https://img.shields.io/badge/GitHub-View%20Notebook-black?logo=github)](https://github.com/GabrielMissael/diffusion_posterior_tutorial/blob/master/notebooks/diffusion_posterior_sampling.ipynb)

_By: Gabriel Missael Barco, Salma Salhi_

This notebook is a tutorial on Bayesian inverse problems with diffusion priors. We move from PSF deconvolution and super-resolution to source reconstruction in strong gravitational lensing.

**Learning goals**

1. Write inverse problems as likelihood × prior → posterior.
2. Diagnose posterior samples with residuals, posterior means, and uncertainty maps.
3. Extend the same probabilistic logic from linear imaging operators to a fixed-lens strong-lensing forward model.

## Credits And Software

- **Workshop data**: `galaxies.pt`, `mistery_galaxy_obs.pt`, and `true_mistery_galaxy.pt` are reused from the `super-resolution-workshop` teaching materials and copied into this repository for a self-contained tutorial.
- **Pretrained score prior**: the diffusion prior is downloaded from Hugging Face repo `GMissaelBarco/galaxy-prior` and loaded with `ScoreModel`.
- **`score-models`**: provides the score-based diffusion model interface, SDE object, score evaluation, and Tweedie denoising used in the posterior samplers.
- **`caustics`**: provides the differentiable strong-lensing physics engine used here for the fixed-lens ray-tracing simulator.
- **Tutorial scope**: the lensing notebook section is a lightweight educational wrapper around the broader research workflow, keeping only source inference with a fixed lens.

## 1. Training data

The model was trained on mock galaxy images produced by applying dust–radiativetransfer post-processing to galaxies in the TNG cosmological magneto-hydrodynamical simulations (<https://www.tng-project.org/>):

> Connor Bottrell, Hassen M. Yesuf, Gergö Popping, Kiyoaki Christopher Omori, Shenli Tang, Xuheng Ding, Annalisa Pillepich, Dylan Nelson, Lukas Eisert, Hua Gao, Andy D. Goulding, Boris S. Kalita, Wentao Luo, Jenny E. Greene, Jingjing Shi, and John D. Silverman. IllustrisTNG in the HSC-SSP: image data release and the major role of mini mergers as drivers of asymmetry and star formation. MNRAS, 527(3):6506-6539, January 2024. [doi:10.1093/mnras/stad2971](https://doi.org/10.1093/mnras/stad2971)

We downsample to $64\times 64$, convert to $\mu \text{Jy} / sr$, and normalize each image to $[0,1]$. If you use this model, please **cite data source** appropriately.


## Bayesian Setup

We use the observation model

$$
y = A(x) + n, \qquad n \sim \mathcal{N}(0, \Sigma_n),
$$

where `A` is either a linear imaging operator or a fixed-lens forward model. A pretrained diffusion model supplies an approximation to the prior score

$$
\nabla_{x_t} \log p_t(x_t).
$$

Posterior sampling replaces the prior score by the posterior score

$$
\nabla_{x_t} \log p_t(x_t \mid y)
= \nabla_{x_t} \log p_t(x_t) + \nabla_{x_t} \log p_t(y \mid x_t).
$$

For the lensing section we instead exploit the fixed-lens linear operator $A_\eta$ and use a CLA-style approximation to the likelihood score, matching the source-only workflow emphasized in class.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from typing import Optional

REPO_NAME = "diffusion_posterior_tutorial"
REPO_URL = "https://github.com/GabrielMissael/diffusion_posterior_tutorial.git"


def in_colab() -> bool:
    return "google.colab" in sys.modules


def find_local_repo_root() -> Optional[Path]:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "diffusion_posterior_tutorial").exists():
            return candidate
    return None


if in_colab():
    REPO_ROOT = Path("/content") / REPO_NAME
    if not REPO_ROOT.exists():
        print("Cloning tutorial repository from GitHub...")
        subprocess.check_call(["git", "clone", "--quiet", REPO_URL, str(REPO_ROOT)])
else:
    REPO_ROOT = find_local_repo_root()
    if REPO_ROOT is None:
        raise RuntimeError(
            "Could not locate the repository root. Run this notebook from a checkout of the tutorial repository or open it in Colab."
        )

SRC_DIR = REPO_ROOT / "src"
os.chdir(REPO_ROOT)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repository root: {REPO_ROOT}")
print(f"Source path added for local imports: {SRC_DIR}")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT)])

!pip3 install --quiet "caskade==0.10.5"
!pip3 install --quiet "caustics==1.2.0"
!pip3 install --quiet git+https://github.com/AlexandreAdam/score_models.git@dev

In [ ]:
import importlib
import math
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display, update_display
from huggingface_hub import snapshot_download
from score_models import ScoreModel
from uuid import uuid4

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "diffusion_posterior_tutorial" or module_name.startswith("diffusion_posterior_tutorial."):
        del sys.modules[module_name]

from diffusion_posterior_tutorial.lensing import LensingSceneConfig, TutorialLensingSimulator
from diffusion_posterior_tutorial.sampling import FixedLensPosteriorSampler, LinearGaussianPosteriorSampler
from diffusion_posterior_tutorial.viz import (
    build_history_scrubber,
    plot_forward_residual_panel,
    plot_history_montage,
    plot_image_grid,
    plot_posterior_summary,
    render_live_frame,
    tensor_to_display,
)
from diffusion_posterior_tutorial.workshop_ops import (
    OBS_NPIX,
    SOURCE_NPIX,
    add_gaussian_noise,
    apply_linear_operator,
    build_psf_downsample_matrix,
    downsample_flux_preserving,
    get_default_device,
    psf_blur,
    set_seed,
)


def display_figure(output_widget, fig):
    output_widget.clear_output(wait=True)
    with output_widget:
        display(fig)
    plt.close(fig)


def make_live_callback(output_widget, *, log_source=True, origin="upper"):
    def _callback(frame):
        fig = render_live_frame(frame, sample_index=0, log_source=log_source, origin=origin)
        display_figure(output_widget, fig)

    return _callback


def sample_prior(
    model,
    *,
    n_samples=5,
    steps=80,
    device=None,
    visualize=True,
    visualize_stride=1,
    log_scale=True,
    origin="upper",
):
    device = device or get_default_device()
    image_size = SOURCE_NPIX

    x = model.sde.prior((1, image_size, image_size)).sample([n_samples]).to(device)
    t = torch.full((n_samples,), float(model.sde.T), device=device)
    dt = -(model.sde.T - model.sde.epsilon) / steps
    x_mean = x

    display_id = f"prior-sampling-{uuid4()}"
    display_initialized = False

    def _show(batch, step, time_value, title_prefix="Prior sampling"):
        nonlocal display_initialized
        images = batch.detach().cpu()[:, 0]

        fig, axes = plt.subplots(1, n_samples, figsize=(3 * n_samples, 3), squeeze=False)
        for i, ax in enumerate(axes[0]):
            ax.imshow(
                tensor_to_display(images[i], log_scale=log_scale),
                cmap="magma",
                origin=origin,
            )
            ax.set_title(f"Sample {i + 1}")
            ax.axis("off")

        fig.suptitle(f"{title_prefix}: step {step}/{steps}, t = {time_value:.3f}", fontsize=12)
        fig.tight_layout()

        if not display_initialized:
            display(fig, display_id=display_id)
            display_initialized = True
        else:
            update_display(fig, display_id=display_id)

        plt.close(fig)

    if visualize:
        with torch.no_grad():
            estimate = model.tweedie(t, x_mean).view(n_samples, 1, image_size, image_size).clamp_min(0.0)
        _show(estimate, step=0, time_value=float(t[0].item()))

    for step in range(steps):
        t_old = t
        t_new = t + dt

        with torch.no_grad():
            g1 = model.sde.diffusion(t_old, x)
            f1 = model.sde.drift(t_old, x)
            s1 = model.score(t_old, x)
        drift1 = f1 - g1.square() * s1
        dw = torch.randn_like(x) * (-dt) ** 0.5
        x_e = x + drift1 * dt + g1 * dw

        with torch.no_grad():
            g2 = model.sde.diffusion(t_new, x_e)
            f2 = model.sde.drift(t_new, x_e)
            s2 = model.score(t_new, x_e)
        drift2 = f2 - g2.square() * s2

        x = x + 0.5 * (drift1 + drift2) * dt + g1 * dw
        x_mean = x - g1 * dw
        t = t_new

        if visualize and ((step + 1) % visualize_stride == 0 or step == steps - 1):
            with torch.no_grad():
                estimate = model.tweedie(t, x_mean).view(n_samples, 1, image_size, image_size).clamp_min(0.0)
            _show(estimate, step=step + 1, time_value=float(t[0].item()))

    with torch.no_grad():
        t0 = torch.full((n_samples,), float(model.sde.t_min), device=device)
        samples = model.tweedie(t0, x_mean).view(n_samples, 1, image_size, image_size).clamp_min(0.0)

    if visualize:
        _show(samples, step=steps, time_value=float(t0[0].item()), title_prefix="Final prior samples")

    return samples.detach().cpu()

DEVICE = get_default_device()
DATA_DIR = REPO_ROOT / "data"
MODEL_DIR = REPO_ROOT / "model" / "galaxy_prior"
MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)

if not MODEL_DIR.exists():
    snapshot_download(
        repo_id="GMissaelBarco/galaxy-prior",
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False,
    )

model = ScoreModel(path=str(MODEL_DIR)).to(DEVICE)
model.load()
model.eval()

galaxies = torch.load(DATA_DIR / "galaxies.pt", map_location=DEVICE).float()
mystery_obs = torch.load(DATA_DIR / "mistery_galaxy_obs.pt", map_location=DEVICE).float()
true_mystery = torch.load(DATA_DIR / "true_mistery_galaxy.pt", map_location=DEVICE).float()

print(f"Using device: {DEVICE}")
print(f"Galaxy batch shape: {tuple(galaxies.shape)}")


## 1. Warm-Up: Forward Degradation Model

For the telescope-style warm-up we use

$$
y = D P x + n, \qquad n \sim \mathcal{N}(0, \sigma_n^2 I),
$$

where `P` is PSF convolution and `D` is flux-preserving downsampling. The key point is not that the image looks worse, but that information has been irreversibly redistributed or lost.


In [ ]:
galaxy_selector = widgets.Dropdown(
    options=[(f"Galaxy {idx}", idx) for idx in range(min(5, galaxies.shape[0]))],
    value=0,
    description="Reference galaxy",
)
source_preview_out = widgets.Output()


def update_source_preview(change=None):
    del change
    image = galaxies[int(galaxy_selector.value)]
    fig = plot_image_grid(
        [image],
        titles=["Chosen source image"],
        log_scale=True,
        origin="upper",
        figsize=(3.5, 3.5),
    )
    display_figure(source_preview_out, fig)


galaxy_selector.observe(update_source_preview, names="value")
display(widgets.VBox([galaxy_selector, source_preview_out]))
update_source_preview()


In [ ]:
warmup_sigma_psf = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.05, description="PSF sigma")
warmup_downsample = widgets.IntSlider(value=32, min=16, max=64, step=4, description="Pixels")
warmup_sigma_n = widgets.FloatSlider(value=0.03, min=0.0, max=0.12, step=0.005, description="Noise")
warmup_preview_out = widgets.Output()


def update_warmup_preview(change=None):
    del change
    sigma_psf = float(warmup_sigma_psf.value)
    size = int(warmup_downsample.value)
    sigma_n = float(warmup_sigma_n.value)
    clean = galaxies[int(galaxy_selector.value):int(galaxy_selector.value) + 1]
    blurred = psf_blur(clean, sigma_psf)
    degraded = downsample_flux_preserving(blurred, size)
    noisy = add_gaussian_noise(degraded, sigma_n)

    fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
    panels = [clean[0], blurred[0], degraded[0], noisy[0]]
    titles = ["Clean source", "PSF blur", "Blur + downsample", "Observed image"]
    for ax, panel, title in zip(axes, panels, titles):
        ax.imshow(tensor_to_display(panel, log_scale=title == "Clean source"), cmap="magma", origin="upper")
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    display_figure(warmup_preview_out, fig)


for widget in [galaxy_selector, warmup_sigma_psf, warmup_downsample, warmup_sigma_n]:
    widget.observe(update_warmup_preview, names="value")

display(widgets.VBox([warmup_sigma_psf, warmup_downsample, warmup_sigma_n, warmup_preview_out]))
update_warmup_preview()


## 2. Prior Samples From The Diffusion Prior

Before conditioning on any data, it is useful to look at what the learned prior generates on its own. The row below shows five unconditional draws from the pretrained galaxy diffusion prior.


In [ ]:
prior_samples = sample_prior(model, n_samples=5, steps=80, device=DEVICE)
fig = plot_image_grid(
    [prior_samples[idx, 0] for idx in range(5)],
    titles=[f"Prior sample {idx + 1}" for idx in range(5)],
    log_scale=True,
    origin="upper",
    figsize=(15, 3),
)
display(fig)
plt.close(fig)


## 3. Warm-Up Posterior Sampling

We now condition on the degraded observation and visualize how one posterior sample evolves at every diffusion step while the full batch is used to estimate posterior means and uncertainties.


In [ ]:
warmup_true = galaxies[int(galaxy_selector.value):int(galaxy_selector.value) + 1]
warmup_size = int(warmup_downsample.value)
warmup_A = build_psf_downsample_matrix(
    (SOURCE_NPIX, SOURCE_NPIX),
    sigma_psf=float(warmup_sigma_psf.value),
    downsample_size=warmup_size,
    device=DEVICE,
)
warmup_y_clean = apply_linear_operator(warmup_true, warmup_A, output_shape=(warmup_size, warmup_size))
warmup_y_obs = add_gaussian_noise(warmup_y_clean, float(warmup_sigma_n.value))

warmup_live_out = widgets.Output()
display(widgets.HTML("<b>Live warm-up trajectory for one sample</b>"))
display(warmup_live_out)

warmup_sampler = LinearGaussianPosteriorSampler(
    observation=warmup_y_obs[0],
    A=warmup_A,
    model=model,
    sigma_n=float(warmup_sigma_n.value),
    device=DEVICE,
)
warmup_result = warmup_sampler.run(
    n_samples=4,
    steps=80,
    progress=True,
    record_history=True,
    history_stride=10,
    max_history_samples=1,
    live_callback=make_live_callback(warmup_live_out, log_source=True, origin="upper"),
    live_stride=1,
)
print("Warm-up sampling complete.")


In [ ]:
def warmup_forward(batch: torch.Tensor) -> torch.Tensor:
    return apply_linear_operator(batch, warmup_A, output_shape=(warmup_size, warmup_size))

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(tensor_to_display(warmup_true[0], log_scale=True), cmap="magma", origin="upper")
axes[0].set_title("True source")
axes[0].axis("off")
axes[1].imshow(tensor_to_display(warmup_y_obs[0], log_scale=False), cmap="magma", origin="upper")
axes[1].set_title("Observed data")
axes[1].axis("off")
plt.tight_layout()
plt.show()

fig = plot_posterior_summary(warmup_result, truth=warmup_true[0], log_scale=True, origin="upper")
display(fig)
plt.close(fig)

fig = plot_forward_residual_panel(
    warmup_result.samples,
    observation=warmup_y_obs[0],
    forward_model=warmup_forward,
    noise_sigma=float(warmup_sigma_n.value),
    max_show=1,
    origin="upper",
)
display(fig)
plt.close(fig)

fig = plot_history_montage(warmup_result.history, sample_index=0, max_frames=6, log_source=True, origin="upper")
display(fig)
plt.close(fig)

display(build_history_scrubber(warmup_result.history, sample_index=0, log_source=True, origin="upper"))


## 4. Mystery Object Challenge

This is deliberately out-of-distribution. The question is no longer “does the model sharpen images?” but rather “how does the posterior behave when the learned prior is imperfect for the underlying object class?”


In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for idx in range(12):
    ax = axes[idx // 6, idx % 6]
    ax.imshow(tensor_to_display(mystery_obs[idx], log_scale=False), cmap="magma", origin="upper")
    ax.set_title(f"Exposure {idx + 1}")
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
mystery_sigma_n = 0.10
mystery_sigma_psf = 0.10
mystery_A = build_psf_downsample_matrix(
    (SOURCE_NPIX, SOURCE_NPIX),
    sigma_psf=mystery_sigma_psf,
    downsample_size=SOURCE_NPIX,
    device=DEVICE,
)

mystery_live_out = widgets.Output()
display(widgets.HTML("<b>Live mystery-object trajectory for one sample</b>"))
display(mystery_live_out)

mystery_sampler = LinearGaussianPosteriorSampler(
    observation=mystery_obs,
    A=mystery_A,
    model=model,
    sigma_n=mystery_sigma_n,
    device=DEVICE,
)
mystery_result = mystery_sampler.run(
    n_samples=4,
    steps=60,
    progress=True,
    record_history=True,
    history_stride=10,
    max_history_samples=1,
    live_callback=make_live_callback(mystery_live_out, log_source=True, origin="upper"),
    live_stride=1,
)


def mystery_forward(batch: torch.Tensor) -> torch.Tensor:
    return apply_linear_operator(batch, mystery_A, output_shape=(SOURCE_NPIX, SOURCE_NPIX))

fig = plot_posterior_summary(mystery_result, truth=true_mystery[0], log_scale=True, origin="upper")
display(fig)
plt.close(fig)

fig = plot_forward_residual_panel(
    mystery_result.samples,
    observation=mystery_obs[0],
    forward_model=mystery_forward,
    noise_sigma=mystery_sigma_n,
    max_show=1,
    origin="upper",
)
display(fig)
plt.close(fig)

fig = plot_history_montage(mystery_result.history, sample_index=0, max_frames=6, log_source=True, origin="upper")
display(fig)
plt.close(fig)

display(build_history_scrubber(mystery_result.history, sample_index=0, log_source=True, origin="upper"))


## 5. Transition To Strong Lensing

The probabilistic structure does not change. We still condition on noisy observations through a forward model,

$$
y = \mathcal{H}(\mathcal{L}_\eta(s)) + n,
$$

but now the operator contains ray tracing through a fixed lens model `\eta`. For this notebook the target distribution is

$$
p(s \mid y, \eta) \propto p(y \mid s, \eta)\, p_{\mathrm{prior}}(s).
$$

We keep the lens fixed, sample only the source, and use a compact `caustics` wrapper for the simulator.


In [ ]:
lensing_simulator = TutorialLensingSimulator(device=DEVICE)
lensing_state = {}

theta_slider = widgets.FloatSlider(value=1.35, min=0.8, max=2.2, step=0.05, description="Einstein r")
phi_slider = widgets.FloatSlider(value=0.4, min=0.0, max=float(np.pi), step=0.05, description="Angle")
q_slider = widgets.FloatSlider(value=0.82, min=0.7, max=1.0, step=0.01, description="Axis ratio")
sx_slider = widgets.FloatSlider(value=0.08, min=-0.35, max=0.35, step=0.01, description="Source x")
sy_slider = widgets.FloatSlider(value=-0.05, min=-0.35, max=0.35, step=0.01, description="Source y")
noise_slider = widgets.FloatSlider(value=0.03, min=0.01, max=0.08, step=0.005, description="Noise")
lens_preview_out = widgets.Output()


def current_lensing_setup():
    source = galaxies[int(galaxy_selector.value)]
    config = LensingSceneConfig(
        theta_e=float(theta_slider.value),
        phi=float(phi_slider.value),
        axis_ratio=float(q_slider.value),
        source_x=float(sx_slider.value),
        source_y=float(sy_slider.value),
        noise_sigma=float(noise_slider.value),
    )
    clean = lensing_simulator.simulate_clean(source.unsqueeze(0), config)
    observed = lensing_simulator.observe(source.unsqueeze(0), config, add_noise=True)
    lensing_state.update({
        "source": source,
        "clean": clean[0],
        "observation": observed[0],
        "config": config,
    })
    return lensing_state


def update_lensing_preview(change=None):
    del change
    state = current_lensing_setup()
    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2))
    axes[0].imshow(tensor_to_display(state["source"], log_scale=True), cmap="magma", origin="lower")
    axes[0].set_title("Chosen source plane")
    axes[1].imshow(tensor_to_display(state["clean"], log_scale=False), cmap="magma", origin="lower")
    axes[1].set_title("Lensed image")
    axes[2].imshow(tensor_to_display(state["observation"], log_scale=False), cmap="magma", origin="lower")
    axes[2].set_title("Observed image")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    display_figure(lens_preview_out, fig)


for widget in [galaxy_selector, theta_slider, phi_slider, q_slider, sx_slider, sy_slider, noise_slider]:
    widget.observe(update_lensing_preview, names="value")

display(widgets.HTML("Adjust the lens configuration here. Then execute the next cell to run posterior sampling for this fixed setup."))
display(widgets.VBox([theta_slider, phi_slider, q_slider, sx_slider, sy_slider, noise_slider, lens_preview_out]))
update_lensing_preview()


## 6. Lensing Posterior Sampling With A Diffusion Prior

For the lensing example we use a **CLA-style** guidance term rather than Tweedie plug-in guidance. The fixed-lens simulator is linear in the source brightness, so we build the corresponding operator $A_\eta$ and use the convolved-likelihood approximation with a diagonal covariance approximation, mirroring the logic used in the workshop and in the research code.

As before, the sampler records selected timesteps. During sampling we display one evolving sample, and after sampling we summarize the posterior with means, standard deviations, residuals, and posterior-predictive diagnostics.


In [ ]:
state = current_lensing_setup()
source_truth = state["source"].unsqueeze(0)
observation = state["observation"].unsqueeze(0)
config = state["config"]

print("Building the fixed-lens operator A_eta for the chosen configuration...")
lensing_A = lensing_simulator.build_matrix(config, chunk_size=128)

lensing_live_out = widgets.Output()
display(widgets.HTML("<b>Live lensing trajectory for one sample</b>"))
display(lensing_live_out)

lensing_sampler = FixedLensPosteriorSampler(
    observation=observation,
    A=lensing_A,
    model=model,
    sigma_n=float(config.noise_sigma),
    device=DEVICE,
)
lensing_result = lensing_sampler.run(
    n_samples=4,
    steps=250,
    progress=True,
    record_history=True,
    history_stride=25,
    max_history_samples=1,
    live_callback=make_live_callback(lensing_live_out, log_source=True, origin="lower"),
    live_stride=1,
)

predictive = apply_linear_operator(
    lensing_result.samples[:, 0],
    lensing_A,
    output_shape=(OBS_NPIX, OBS_NPIX),
).detach().cpu()


def lensing_forward(batch: torch.Tensor) -> torch.Tensor:
    return apply_linear_operator(batch, lensing_A, output_shape=(OBS_NPIX, OBS_NPIX))

fig = plot_posterior_summary(lensing_result, truth=source_truth[0], log_scale=True, origin="lower")
display(fig)
plt.close(fig)

fig = plot_forward_residual_panel(
    lensing_result.samples,
    observation=observation[0],
    forward_model=lensing_forward,
    noise_sigma=float(config.noise_sigma),
    max_show=1,
    origin="lower",
)
display(fig)
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2))
axes[0].imshow(tensor_to_display(observation[0], log_scale=False), cmap="magma", origin="lower")
axes[0].set_title("Observed image")
axes[1].imshow(tensor_to_display(predictive.mean(dim=0), log_scale=False), cmap="magma", origin="lower")
axes[1].set_title("Posterior predictive mean")
axes[2].imshow(tensor_to_display(predictive.std(dim=0), log_scale=False), cmap="viridis", origin="lower")
axes[2].set_title("Posterior predictive std")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

fig = plot_history_montage(lensing_result.history, sample_index=0, max_frames=6, log_source=True, origin="lower")
display(fig)
plt.close(fig)

display(build_history_scrubber(lensing_result.history, sample_index=0, log_source=True, origin="lower"))


## 7. Exercises And Discussion

1. Increase the PSF width in the warm-up problem. Which source features disappear first from the posterior mean, and which ones remain visible only in the posterior spread?
2. In the mystery-object example, where do the residuals suggest a mismatch between the prior and the data?
3. In the lensing section, move the source toward and away from the caustic. How do the posterior standard deviation and the image-plane residuals respond?
4. Change the lens axis ratio and orientation. Which source-plane structures become strongly constrained by multiple images, and which remain prior-dominated?
5. If the lens model were misspecified, what signatures would you expect in the normalized residual maps and in the posterior predictive spread?
